# 4.5 Shared Evidence Graph Between Agents: Does a Graph of Claims Beat Message Passing and a Single Agent?

*Formerly notebook 29.*

> **Outputs in this copy come from the confidence run** (`SMASH_MODE=batch`): 1,000 questions per benchmark, 53,995
> calls through the Batch API, $6.74. Task 4.5. They are reported in `RESULTS-REPORT.md` and the v2 paper.

**Presentation map:** Task 4.5 (shared graph between agents), Slide 5 (provenance: every claim points to its source
passage). Paper v2, `PLAN.md` §5, row 5.5.

## Protocol (frozen before the confidence run)

**Research question.** Do two agents that share one evidence graph of claims answer multi-hop questions more accurately,
and with fewer contradictions, than the same two agents exchanging messages, and than a single agent given the same
total token budget?

**Hypothesis.** H1: the shared graph (b) has a higher answer F1 than message passing (a), because the verifier's verdicts
are stored as claim statuses and the final answer is read only from supported (and still open) claims, never from
refuted ones. H2: (b) has fewer final answers with a contradiction than (a). Published work (Tran & Kiela 2026, MuSiQue)
predicts that a single agent at equal token budget (c) matches or beats multi-agent set-ups; a positive result for the
graph is credible only if (b) also beats (c).

**Data.** MuSiQue (answerable, dev) and 2WikiMultiHopQA (dev), the 1,000 questions each of Notebook 1.4a, over its passage
pools (11,656 and 6,119 Wikipedia passages) and its passage and question embeddings (`text-embedding-3-small`, read from
Notebook 1.4a's cache, nothing is re-embedded). n = 2,000 questions (test run: the first 25 per benchmark, fixed order).

**Search tool (same for every condition).** Dense search over the passage pool with the query embedding; each search
returns the 4 most similar passages not retrieved before in the episode. The first search uses the question itself;
**at most 4 searches per episode** in every condition (up to 16 passages), so all conditions can see the same amount
of evidence.

**Conditions (one model, `gpt-6-luna`, no reasoning, temperature 0):**
- **(a) Message passing.** A retriever agent and a verifier agent. Schedule (8 calls): R1, R2, V1, R3, V2, R4, V3,
  final. The retriever reads all passages retrieved so far plus the message transcript, writes a message with its
  findings (each citing a passage `[P#]`) and the next search query (R1–R3). The verifier reads the same passages and
  transcript and replies whether each finding is supported, refuted or open and what is missing. The **final answerer**
  reads the question and the full transcript (no passages).
- **(b) Shared evidence graph.** The same two agents, the same schedule, passages and searches, but the channel is one
  graph: claim nodes (text, author agent, round, status `supported | refuted | open`, verifier's reason), a `source`
  edge from each claim to its passage node (provenance), `builds_on` edges between claims, and open-question nodes
  written by the verifier. The retriever adds claims; the verifier marks them. The **final answerer** reads the
  question and the supported claims (and, separately labelled, the still-open claims); refuted claims are not shown.
- **(c) Single agent at equal total token budget.** One agent with the same search tool and the same 4-search limit.
  Each step it reads all passages and its own notes, writes notes (facts with `[P#]`, each checked against its
  passage), and chooses `search`, `continue` or `answer`. Its **budget per question is the mean total tokens
  (input + output) that (a) and (b) spent on that question**; it is told the remaining budget, and the last call is
  forced to answer when the next step would leave too little for the answer (at most 12 calls). Budgets are reported.

All three give the same final output: a justification of at most three sentences and a short answer.

**Primary metric.** Answer token F1 against the gold answer (and aliases), as in Notebook 1.4a.
**Primary comparisons:** (b) − (a) (the research question) and (b) − (c) (the strong baseline); paired 95% bootstrap
over questions (5,000 resamples). Verdict "graph better" only if both intervals exclude 0 on the positive side.

**Secondary metrics.** Exact match; share of final answers with a contradiction (LLM judge, `gpt-6-luna`: one structured
call per final answer sees the union of the passages the three conditions retrieved for that question and lists the
statements a passage contradicts, plus whether the output contradicts itself); total tokens and calls per question,
by agent; searches used; claim statuses in (b). All results also by hop count (MuSiQue 2/3/4-hop) and question type
(2Wiki).

**Threats.** (1) The equal budget is matched per question but the single agent may stop early; its actual tokens are
reported next to the budget. (2) The contradiction judge is the same model as the agents; a second judge (gpt-4.1) on a
500-item subsample should be run on the stage `nb29_contradiction_judge` (as `second_judge.py` does for v1). (3) One prompt per role, fixed before the test run and not tuned on
test items. (4) The retriever is dense only (Notebook 1.4a's vector index); the graph here is a coordination medium, not a
retrieval index. (5) Contradictions are judged against retrieved passages, not against all of Wikipedia.

**Protocol changes after the test run:**
1. *Added a secondary condition (c′) "single agent, all 4 searches required".* Reason: in the test run (c) answered
   after 2–3 calls and about 2.3 searches (of 4) and used about a quarter of its token budget, so it saw less evidence
   than (a) and (b). (c′) is (c) with one change: while searches are left, every step must search (it may not answer or
   `continue` before the 4th search); after that it behaves as (c). Same token cap (the per-question budget of (c)), same
   call cap, same prompts otherwise. **The frozen primary contrasts are unchanged** ((b) − (a), (b) − (c)); (b) − (c′) is
   reported as a **secondary** contrast. Its contradiction judge (stage `nb29_contradiction_judge_c4`) sees the union of
   the passages of (a), (b), (c) **and** (c′); the frozen judge of (a)–(c) is unchanged (union of (a)–(c)).

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import json, os, re, string, time
from collections import Counter, defaultdict
from typing import Literal
import numpy as np
import networkx as nx
from pydantic import BaseModel
from smash_batch import LLM, mode_from_env, wilson, paired_boot, EMBED_PRICE

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
BASE = "data/benchmarks/musique_2wiki"
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))   # Notebook 1.4a's cache: its embeddings are read, not rebuilt
N_Q = 25 if TEST else 1000           # questions per benchmark (test mode: the first 25 of the same fixed order)

def load(name):
    qs = json.load(open(f"{BASE}/{name}.json"))
    corpus = json.load(open(f"{BASE}/{name}_corpus.json"))
    return qs, corpus

DATA = {}
for name in ("musique", "2wikimultihopqa"):
    qs, corpus = load(name)
    for q in qs:                                     # one schema for both benchmarks (as in Notebook 1.5)
        if name == "musique":
            q["qid"], q["golds"] = q["id"], [q["answer"]] + q.get("answer_aliases", [])
            q["group"] = q["id"].split("hop")[0] + "-hop"
        else:
            q["qid"], q["golds"] = q["_id"], [q["answer"]]
            q["group"] = q["type"]
    DATA[name] = {"questions": qs[:N_Q], "corpus": corpus, "name": name}
    print(f"{name}: {len(qs):,} questions ({N_Q} used), {len(corpus):,} passages | groups used:",
          dict(Counter(q["group"] for q in qs[:N_Q])))
print("MODE =", MODE)

musique: 1,000 questions (1000 used), 11,656 passages | groups used: {'2-hop': 518, '3-hop': 316, '4-hop': 166}
2wikimultihopqa: 1,000 questions (1000 used), 6,119 passages | groups used: {'compositional': 413, 'comparison': 244, 'bridge_comparison': 235, 'inference': 108}
MODE = batch


## The passage pools and embeddings of Notebook 1.4a (read from its cache)

Notebook 1.4a's code, unchanged (as in Notebook 1.5), so that the embedding requests match its cache and nothing is
recomputed. In test mode the pool is the paragraphs that come with the 25 questions, as in Notebooks 1.4a and 1.5.

In [2]:
for name, d in DATA.items():
    passages = [p["title"] + "\n" + p["text"] for p in d["corpus"]]
    if TEST:
        keep = set()
        for q in d["questions"]:
            if name == "musique":
                keep |= {p["title"] + "\n" + p["paragraph_text"] for p in q["paragraphs"]}
            else:
                keep |= {t + "\n" + " ".join(s) for t, s in q["context"]}
        passages = [p for p in passages if p in keep]
    d["passages"] = passages
    d["P"] = llm.embed(f"{name}_passages", d["passages"])
    d["Q"] = llm.embed(f"{name}_questions", [q["question"] for q in d["questions"]])
    print(f"{name}: {len(passages):,} passages in the pool; passage matrix {d['P'].shape}, questions {d['Q'].shape}")

musique: 11,656 passages in the pool; passage matrix (11656, 1536), questions (1000, 1536)
2wikimultihopqa: 6,119 passages in the pool; passage matrix (6119, 1536), questions (1000, 1536)


## Episodes, the search tool and the two channels

An **episode** is one condition on one question. It holds the passages retrieved so far (labelled `P1`, `P2`, … in the
order they arrived), the token and call counts per agent, and its channel: a message transcript in (a), a
`networkx` graph in (b), the agent's own notes in (c). Search queries written by the agents are embedded in one call
per step (cached as `nb29_queries_<step>`).

In [3]:
K_SEARCH, MAX_SEARCHES, N_ROUNDS, MAX_SINGLE = 4, 4, 4, 12
CONDS = {"mp": "(a) messages", "graph": "(b) shared graph", "single": "(c) single agent"}   # frozen primary conditions
EXTRA = {"single4": "(c') 4 searches"}                     # secondary condition added after the test run
COLS = {**CONDS, **EXTRA}
STAGES, EMBEDS = [], []
ntok = lambda s: len(s) // 4 + 1           # rough token estimate, used only for the single agent's budget check

def run(stage, reqs):
    if stage not in STAGES:
        STAGES.append(stage)
    return llm.run(stage, reqs) if reqs else {}

def parsed(r, cls):
    try:
        return cls.model_validate(json.loads(r.get("text") or "")).model_dump()
    except Exception:
        return None

def question(st):
    return DATA[st["bench"]]["questions"][st["qi"]]["question"]

def search(st, qvec, text):
    s = DATA[st["bench"]]["P"] @ qvec
    seen = set(st["pids"])
    new = [int(i) for i in np.argsort(-s)[:K_SEARCH + len(seen)] if int(i) not in seen][:K_SEARCH]
    st["pids"] += new; st["searches"] += 1; st["queries"].append(text)

def search_all(queries, tag):
    # queries: {episode key: query text}; one embedding call for the step, then one search per episode.
    keys = sorted(queries)
    if not keys:
        return
    EMBEDS.append(f"nb29_queries_{tag}")
    V = llm.embed(f"nb29_queries_{tag}", [queries[k] for k in keys], cap=500)
    for k, v in zip(keys, V):
        search(EP[k], v, queries[k])

def passages_block(st):
    d = DATA[st["bench"]]
    return "\n\n".join(f"[P{n + 1}] {d['passages'][p]}" for n, p in enumerate(st["pids"]))

def account(st, r, agent):
    t = r.get("in", 0) + r.get("out", 0)
    st["tokens"] += t; st["calls"] += 1; st["agent_tokens"][agent] += t

def new_episode(cond, bench, qi):
    st = {"cond": cond, "bench": bench, "qi": qi, "pids": [], "queries": [], "searches": 0, "tokens": 0, "calls": 0,
          "agent_tokens": Counter(), "final": None, "msgs": [], "notes": [], "G": nx.DiGraph(), "n_claims": 0}
    search(st, DATA[bench]["Q"][qi], question(st))           # first search: the question (no LLM call)
    return st

EP = {f"{c}|{b}|{qi}": new_episode(c, b, qi) for c in COLS for b, d in DATA.items() for qi in range(len(d["questions"]))}
print(f"{len(EP):,} episodes; every one starts with the {K_SEARCH} passages most similar to the question")

8,000 episodes; every one starts with the 4 passages most similar to the question


### The two channels of the two-agent team

- **(a)** `transcript()`: every message so far, labelled with its author and round.
- **(b)** `graph_view()`: the claim graph as text. Each claim line shows its id, status and the verifier's reason, its
  author and round, its source passage (label and title) and the claims it builds on; then the verifier's latest open
  questions. The final answerer gets `graph_view(final=True)`: supported claims, then open claims, no refuted claims.

In [4]:
def transcript(st):
    if not st["msgs"]:
        return "Messages so far: none yet."
    return "Messages so far:\n" + "\n\n".join(f"[{who}, round {r}] {txt}" for who, r, txt in st["msgs"])

def claim_line(st, c):
    G, a = st["G"], st["G"].nodes[c]
    src = next((v for _, v, e in G.out_edges(c, data=True) if e["rel"] == "source"), None)
    deps = [v for _, v, e in G.out_edges(c, data=True) if e["rel"] == "builds_on"]
    title = DATA[st["bench"]]["passages"][G.nodes[src]["pid"]].split("\n")[0] if src else ""
    status = a["status"] + (f": {a['reason']}" if a["reason"] else "")
    meta = f"by {a['author']}, round {a['round']}; source " + (f'{src} "{title}"' if src else "none")
    if deps:
        meta += "; builds on " + ", ".join(deps)
    return f"{c} [{status}] ({meta}) {a['text']}"

def claims(st, status=None):
    return [n for n, a in st["G"].nodes(data=True) if a["kind"] == "claim" and status in (None, a["status"])]

def graph_view(st, final=False):
    if final:
        sup, opn = claims(st, "supported"), claims(st, "open")
        return ("Supported claims (checked by the verifier against their source passages):\n" +
                ("\n".join(claim_line(st, c) for c in sup) or "none") +
                "\n\nOpen claims (not yet verified):\n" + ("\n".join(claim_line(st, c) for c in opn) or "none"))
    cl = claims(st)
    qs = [a for _, a in st["G"].nodes(data=True) if a["kind"] == "question"]
    last = max((a["round"] for a in qs), default=None)
    return ("Shared evidence graph.\nClaims:\n" + ("\n".join(claim_line(st, c) for c in cl) or "none yet") +
            "\nOpen questions from the verifier:\n" +
            ("\n".join(f"- {a['text']}" for a in qs if a["round"] == last) or "none yet"))

def label_to_pid(st, s):
    m = re.search(r"P?(\d+)", str(s))
    n = int(m.group(1)) - 1 if m else -1
    return n if 0 <= n < len(st["pids"]) else None

## Prompts (fixed before the test run)

The role descriptions are identical in (a) and (b); only the channel and the output format differ. The final answerer
and the single agent use the same answer format.

In [5]:
class MsgOut(BaseModel):
    message: str
    next_query: str

class NewClaim(BaseModel):
    text: str
    source: str
    builds_on: list[str]

class ClaimsOut(BaseModel):
    new_claims: list[NewClaim]
    next_query: str

class VerifierMsg(BaseModel):
    message: str

class Mark(BaseModel):
    claim_id: str
    status: Literal["supported", "refuted", "open"]
    reason: str

class VerifierMarks(BaseModel):
    marks: list[Mark]
    open_questions: list[str]

class FinalAnswer(BaseModel):
    justification: str
    answer: str

class SingleStep(BaseModel):
    notes: str
    action: Literal["search", "continue", "answer"]
    query: str
    justification: str
    answer: str

RET_SYS = ("You are the retriever agent of a two-agent team that answers a multi-hop question from Wikipedia passages. "
           "A search tool returns passages. You read the passages, report the facts that lead step by step to the "
           "answer, and choose the next search query to find the fact that is still missing. A verifier agent checks "
           "your findings against the passages. Report only what the passages state, and cite the passage of every "
           "fact by its label (P1, P2, ...).")
VER_SYS = ("You are the verifier agent of a two-agent team that answers a multi-hop question from Wikipedia passages. "
           "You do not search. You check each finding of the retriever against the passages: 'supported' if its "
           "source passage states it; 'refuted' if the passages contradict it or its source does not state it; 'open' "
           "if it cannot be decided yet. You also say which fact is still missing to answer the question. Use only the "
           "passages, not your own knowledge.")
FINAL_SYS = ("You give the final answer of a two-agent team to a multi-hop question. You do not see the passages, only "
             "the team's work below. Answer from that work only. Give a justification of at most three short sentences "
             "and the answer as a short phrase (an entity, date, number or yes/no), not a sentence. If the work does "
             "not establish the answer, give your best guess.")
SINGLE_SYS = ("You answer a multi-hop question from Wikipedia passages, working alone. A search tool returns passages. "
              "At each step you read the passages and write working notes: the facts that lead step by step to the "
              "answer, each citing its passage by label (P1, P2, ...), and a check of each fact against its passage "
              "(supported, refuted or open). Report only what the passages state. Then choose 'search' (give a query "
              "for the fact that is still missing), 'continue' (check your notes again without searching) or 'answer'. "
              "When you answer, give a justification of at most three short sentences and the answer as a short phrase "
              "(an entity, date, number or yes/no), not a sentence; otherwise leave justification and answer empty.")

RET_TASK = {
    "mp": ("Write a message to the verifier with your new findings (short statements, each citing its passage as "
           "[P#]); address what the verifier asked for. Do not repeat findings already reported."),
    "graph": ("Add new claims to the shared graph: each a short self-contained statement, with the label of the passage "
              "that states it (e.g. P3) and the ids of earlier claims it builds on. Address the verifier's open "
              "questions. Do not repeat claims already in the graph.")}
VER_TASK = {
    "mp": ("Reply to the retriever: for each of its findings say whether it is supported, refuted or open, and why "
           "(one short sentence each), then say which fact is still missing to answer the question."),
    "graph": ("Mark the claims that are new or still open (you may also change an earlier mark): status supported, "
              "refuted or open, with a reason of one short sentence. Then add open questions for the facts still "
              "missing to answer the question.")}

def ret_body(st, rnd):
    chan = transcript(st) if st["cond"] == "mp" else graph_view(st)
    q_rule = ("This is the last round: leave next_query empty." if rnd == N_ROUNDS else
              "Then give the next search query (a short query for the missing fact).")
    user = (f"Question: {question(st)}\n\nPassages retrieved so far:\n{passages_block(st)}\n\n{chan}\n\n"
            f"Round {rnd} of {N_ROUNDS}. {RET_TASK[st['cond']]} {q_rule}")
    return LLM.body([{"role": "system", "content": RET_SYS}, {"role": "user", "content": user}], max_tokens=600,
                    schema=MsgOut if st["cond"] == "mp" else ClaimsOut)

def ver_body(st, rnd):
    chan = transcript(st) if st["cond"] == "mp" else graph_view(st)
    user = (f"Question: {question(st)}\n\nPassages retrieved so far:\n{passages_block(st)}\n\n{chan}\n\n"
            f"Verification round {rnd} of {N_ROUNDS - 1}. {VER_TASK[st['cond']]}")
    return LLM.body([{"role": "system", "content": VER_SYS}, {"role": "user", "content": user}], max_tokens=600,
                    schema=VerifierMsg if st["cond"] == "mp" else VerifierMarks)

def final_body(st):
    chan = transcript(st) if st["cond"] == "mp" else graph_view(st, final=True)
    user = f"Question: {question(st)}\n\nThe team's work:\n{chan}"
    return LLM.body([{"role": "system", "content": FINAL_SYS}, {"role": "user", "content": user}], max_tokens=300,
                    schema=FinalAnswer)

## Conditions (a) and (b): the two-agent schedule, one Batch job per step

Both conditions and both benchmarks run in the same step, so the schedule R1, R2, V1, R3, V2, R4, V3, final is eight
Batch jobs. After R1–R3, every episode searches with the retriever's query (the question if the query is empty).

In [6]:
SCHEDULE = [("R", 1), ("R", 2), ("V", 1), ("R", 3), ("V", 2), ("R", 4), ("V", 3), ("F", 0)]
MAS = [k for k, st in EP.items() if st["cond"] in ("mp", "graph")]
UNPARSED = Counter()

def apply_retriever(st, r, rnd):
    if st["cond"] == "mp":
        p = parsed(r, MsgOut) or {"message": r.get("text", ""), "next_query": ""}
        st["msgs"].append(("Retriever", rnd, p["message"].strip()))
        return p, p["next_query"]
    p = parsed(r, ClaimsOut) or {"new_claims": [], "next_query": ""}
    G = st["G"]
    for c in p["new_claims"]:
        st["n_claims"] += 1
        cid = f"C{st['n_claims']}"
        G.add_node(cid, kind="claim", text=c["text"].strip(), author="retriever", round=rnd, status="open",
                   reason="", history=[])
        pid = label_to_pid(st, c["source"])
        if pid is not None:
            G.add_node(f"P{pid + 1}", kind="passage", pid=st["pids"][pid])
            G.add_edge(cid, f"P{pid + 1}", rel="source")
        for dep in c["builds_on"]:
            dep = dep.strip()
            if dep in G and G.nodes[dep]["kind"] == "claim" and dep != cid:
                G.add_edge(cid, dep, rel="builds_on")
    return p, p["next_query"]

def apply_verifier(st, r, rnd):
    if st["cond"] == "mp":
        p = parsed(r, VerifierMsg) or {"message": r.get("text", "")}
        st["msgs"].append(("Verifier", rnd, p["message"].strip()))
        return p
    p = parsed(r, VerifierMarks) or {"marks": [], "open_questions": []}
    G = st["G"]
    for m in p["marks"]:
        c = m["claim_id"].strip()
        if c in G and G.nodes[c]["kind"] == "claim":
            a = G.nodes[c]
            a["history"].append((rnd, m["status"]))
            a["status"], a["reason"] = m["status"], m["reason"].strip()
    for i, oq in enumerate(p["open_questions"]):
        G.add_node(f"Q{rnd}.{i + 1}", kind="question", text=oq.strip(), author="verifier", round=rnd)
    return p

t0 = time.time()
for kind, rnd in SCHEDULE:
    tag = {"R": f"r{rnd}", "V": f"v{rnd}", "F": "final"}[kind]
    build = {"R": lambda st: ret_body(st, rnd), "V": lambda st: ver_body(st, rnd), "F": final_body}[kind]
    res = run(f"nb29_mas_{tag}", {k: build(EP[k]) for k in MAS})
    queries = {}
    for k in MAS:
        st, r = EP[k], res[k]
        account(st, r, {"R": "retriever", "V": "verifier", "F": "final"}[kind])
        if kind == "R":
            p, nq = apply_retriever(st, r, rnd)
            UNPARSED[tag] += parsed(r, MsgOut if st["cond"] == "mp" else ClaimsOut) is None
            if rnd < N_ROUNDS and st["searches"] < MAX_SEARCHES:
                queries[k] = nq.strip() or question(st)
        elif kind == "V":
            apply_verifier(st, r, rnd)
            UNPARSED[tag] += parsed(r, VerifierMsg if st["cond"] == "mp" else VerifierMarks) is None
        else:
            st["final"] = parsed(r, FinalAnswer) or {"justification": "", "answer": r.get("text", "")}
            UNPARSED[tag] += parsed(r, FinalAnswer) is None
    search_all(queries, f"mas_{tag}")
    print(f"step {tag}: {len(MAS):,} calls, {UNPARSED[tag]} unparsed, {len(queries):,} searches")
print(f"two-agent conditions done in {time.time() - t0:.0f} s")

step r1: 4,000 calls, 0 unparsed, 4,000 searches


step r2: 4,000 calls, 0 unparsed, 4,000 searches


step v1: 4,000 calls, 0 unparsed, 0 searches


step r3: 4,000 calls, 0 unparsed, 4,000 searches


step v2: 4,000 calls, 0 unparsed, 0 searches


step r4: 4,000 calls, 0 unparsed, 0 searches


step v3: 4,000 calls, 0 unparsed, 0 searches


step final: 4,000 calls, 0 unparsed, 0 searches
two-agent conditions done in 19 s


## Condition (c): a single agent at equal total token budget

The budget of each question is the mean total tokens that (a) and (b) spent on it (all eight calls). Before every step
the agent's next call is estimated from its prompt (characters / 4 + 300 output tokens); if the tokens used plus twice
that estimate exceed the budget, or the call cap of 12 is reached, the step is a forced answer. Searches are capped at
4, as in (a) and (b); a `search` with no searches left is treated as `continue`.

**(c′), added after the test run (secondary):** the same agent, budget and caps, but while searches are left each step
must write its notes and a search query (schema without an answer field), so it always sees 4 searches of evidence,
as (a) and (b) do, unless the budget runs out first. Stages `nb29_single4_s<step>`.

In [7]:
SINGLE = [k for k, st in EP.items() if st["cond"] in ("single", "single4")]
for k in SINGLE:
    st = EP[k]
    tail = k.split("|", 1)[1]
    st["budget"] = (EP["mp|" + tail]["tokens"] + EP["graph|" + tail]["tokens"]) / 2

class SearchStep(BaseModel):                 # (c') while searches are left: notes and the next query, no answer
    notes: str
    query: str

def must_search(st):
    return st["cond"] == "single4" and st["searches"] < MAX_SEARCHES

def single_messages(st, step, force):
    notes = "\n\n".join(f"[step {s}] {n}" for s, n in st["notes"]) or "none yet"
    left = max(0, int(st["budget"] - st["tokens"]))
    if force:
        tail = "No budget is left for further steps: give the final answer now (justification and answer)."
    elif must_search(st):
        tail = (f"Step {step}. Searches left: {MAX_SEARCHES - st['searches']}. Token budget left: about {left:,} tokens "
                f"(every step re-reads the passages and your notes). You must use all {MAX_SEARCHES} searches before "
                f"answering: write your notes and the next search query.")
    else:
        tail = (f"Step {step}. Searches left: {MAX_SEARCHES - st['searches']}. Token budget left: about {left:,} tokens "
                f"(every step re-reads the passages and your notes). Choose 'search', 'continue' or 'answer'.")
    user = (f"Question: {question(st)}\n\nPassages retrieved so far:\n{passages_block(st)}\n\n"
            f"Your notes so far:\n{notes}\n\n{tail}")
    return [{"role": "system", "content": SINGLE_SYS}, {"role": "user", "content": user}]

def run_single(cond):
    keys, FORCED = [k for k in SINGLE if EP[k]["cond"] == cond], Counter()
    for step in range(1, MAX_SINGLE + 1):
        active = [k for k in keys if EP[k]["final"] is None]
        if not active:
            break
        reqs, forced = {}, set()
        for k in active:
            st = EP[k]
            m = single_messages(st, step, force=False)
            est = ntok(m[0]["content"] + m[1]["content"]) + 300
            if step == MAX_SINGLE or st["tokens"] + 2 * est > st["budget"]:
                forced.add(k)
                reqs[k] = LLM.body(single_messages(st, step, force=True), max_tokens=300, schema=FinalAnswer)
            else:
                reqs[k] = LLM.body(m, max_tokens=700, schema=SearchStep if must_search(st) else SingleStep)
        res = run(f"nb29_{cond}_s{step}", reqs)
        queries = {}
        for k in active:
            st, r = EP[k], res[k]
            account(st, r, "single")
            if k in forced:
                FORCED["forced"] += 1
                st["final"] = parsed(r, FinalAnswer) or {"justification": "", "answer": r.get("text", "")}
                continue
            if must_search(st):
                p = parsed(r, SearchStep)
                if p is None:
                    UNPARSED[f"{cond}_s{step}"] += 1
                    p = {"notes": r.get("text", ""), "query": ""}
                p = {**p, "action": "search"}
            else:
                p = parsed(r, SingleStep)
                if p is None:
                    UNPARSED[f"{cond}_s{step}"] += 1
                    p = {"notes": r.get("text", ""), "action": "continue", "query": "", "justification": "", "answer": ""}
            if p["action"] == "answer":
                FORCED["chosen"] += 1
                st["final"] = {"justification": p["justification"], "answer": p["answer"]}
                continue
            did = p["action"] == "search" and st["searches"] < MAX_SEARCHES
            st["notes"].append((step, p["notes"].strip() + (f"\n(searched: {p['query'].strip()})" if did else "")))
            if did:
                queries[k] = p["query"].strip() or question(st)
        search_all(queries, f"{cond}_s{step}")
        print(f"{cond} step {step}: {len(active):,} active, {len(forced):,} forced to answer, {len(queries):,} searches")
    print(f"{COLS[cond]}: answers chosen by the agent {FORCED['chosen']} | forced by budget or call cap {FORCED['forced']}\n")

run_single("single")      # (c), frozen protocol
run_single("single4")     # (c'), secondary: all 4 searches required

single step 1: 2,000 active, 0 forced to answer, 1,465 searches


single step 2: 1,465 active, 0 forced to answer, 838 searches


single step 3: 842 active, 0 forced to answer, 759 searches
single step 4: 762 active, 0 forced to answer, 4 searches


single step 5: 233 active, 17 forced to answer, 1 searches
single step 6: 122 active, 111 forced to answer, 0 searches
single step 7: 9 active, 6 forced to answer, 0 searches
single step 8: 1 active, 1 forced to answer, 0 searches
(c) single agent: answers chosen by the agent 1865 | forced by budget or call cap 135



single4 step 1: 2,000 active, 0 forced to answer, 2,000 searches


[nb29_single4_s2] 2,000 new requests -> batch (0 cached)


... (147 outputs left out)

[nb29_single4_s8] 6 min: 1 running, 0 queued, 0 results cached


[nb29_single4_s8] batch batch_6af103790a3c81909cf1029c2ee530a5 completed: 1 ok, 0 failed


[nb29_single4_s8] done in 8.1 min


single4 step 8: 1 active, 1 forced to answer, 0 searches
(c') 4 searches: answers chosen by the agent 1774 | forced by budget or call cap 226



## The contradiction judge (one structured call per final answer)

For each question the judge sees the **union** of the passages the three conditions retrieved (so a condition that read
fewer passages is not judged more leniently) and one final output (justification and answer). It lists the statements
that a passage contradicts and says whether the output contradicts itself. A statement the passages do not mention is
not a contradiction. The judge does not see the gold answer or the condition.

In [8]:
class Contradiction(BaseModel):
    statement: str
    passage: str

class ContraVerdict(BaseModel):
    contradictions: list[Contradiction]
    self_contradictory: bool

JUDGE_SYS = ("You check the final output of a question-answering system for contradictions. You receive numbered "
             "Wikipedia passages and the output (a justification and an answer). List every statement of the output "
             "that a passage contradicts (the passage states something incompatible with it), with the label of that "
             "passage, and say whether the output contradicts itself (for example, the justification supports a "
             "different answer than the one given). A statement the passages do not mention is NOT a contradiction. "
             "Use only the passages, not your own knowledge. Return an empty list if nothing is contradicted.")

def union_pids(bench, qi, conds=tuple(CONDS)):
    return list(dict.fromkeys(p for c in conds for p in EP[f"{c}|{bench}|{qi}"]["pids"]))

def judge_body(k):
    st = EP[k]
    conds = tuple(CONDS) if st["cond"] in CONDS else tuple(COLS)     # (c'): the union also includes its passages
    d, pids = DATA[st["bench"]], union_pids(st["bench"], st["qi"], conds)
    out = st["final"]
    msg = ("Passages:\n" + "\n\n".join(f"[J{n + 1}] {d['passages'][p]}" for n, p in enumerate(pids)) +
           f"\n\nQuestion: {question(st)}\n\nOutput:\nJustification: {out['justification']}\nAnswer: {out['answer']}")
    return LLM.body([{"role": "system", "content": JUDGE_SYS}, {"role": "user", "content": msg}], max_tokens=400,
                    schema=ContraVerdict)

JUDGED = run("nb29_contradiction_judge", {k: judge_body(k) for k in EP if EP[k]["cond"] in CONDS})
JUDGED.update(run("nb29_contradiction_judge_c4", {k: judge_body(k) for k in EP if EP[k]["cond"] in EXTRA}))
VERDICT = {k: parsed(JUDGED[k], ContraVerdict) for k in EP}
print(f"{len(JUDGED):,} judge calls, {sum(v is None for v in VERDICT.values())} unparsed; "
      f"passages per judge call: {np.mean([len(union_pids(b, qi)) for b, d in DATA.items() for qi in range(len(d['questions']))]):.1f}")

[nb29_contradiction_judge] 6,000 new requests -> batch (0 cached)


[nb29_contradiction_judge] uploaded input file 1/1 (6,000 requests, 85 MB)


[nb29_contradiction_judge] started batch batch_6af1058f926c8190940bd7815c943c0a


[nb29_contradiction_judge] 1 min: 1 running, 0 queued, 0 results cached


[nb29_contradiction_judge] 6 min: 1 running, 0 queued, 0 results cached


[nb29_contradiction_judge] 11 min: 1 running, 0 queued, 0 results cached


... (46 outputs left out)

[nb29_contradiction_judge_c4] 41 min: 1 running, 0 queued, 0 results cached


[nb29_contradiction_judge_c4] batch batch_6af12f9af77c819086f852d747b78619 completed: 2,000 ok, 0 failed


[nb29_contradiction_judge_c4] done in 43.5 min


8,000 judge calls, 1 unparsed; passages per judge call: 20.4


## Scores per question

In [9]:
def normalize(s):
    s = str(s).lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = Counter(p) & Counter(g); same = sum(common.values())
    if same == 0:
        return 0.0
    pr, rc = same / len(p), same / len(g)
    return 2 * pr * rc / (pr + rc)

def score(k):
    st = EP[k]
    q = DATA[st["bench"]]["questions"][st["qi"]]
    pred = st["final"]["answer"].strip()
    v = VERDICT[k] or {"contradictions": [], "self_contradictory": False}
    return {"f1": max(f1(pred, g) for g in q["golds"]), "em": float(max(normalize(pred) == normalize(g) for g in q["golds"])),
            "contra": float(bool(v["contradictions"]) or v["self_contradictory"]), "n_contra": len(v["contradictions"]),
            "tokens": st["tokens"], "calls": st["calls"], "searches": st["searches"], "passages": len(st["pids"])}

ROWS = {b: [{"group": q["group"], **{c: score(f"{c}|{b}|{qi}") for c in COLS}} for qi, q in enumerate(d["questions"])]
        for b, d in DATA.items()}
print("scored", sum(len(r) for r in ROWS.values()), "questions x", len(COLS), "conditions")

scored 2000 questions x 4 conditions


## Results with 95% confidence intervals

Means per condition; the last four columns are paired bootstrap intervals over questions. The primary metric is F1;
the primary comparisons are (b) − (a) and (b) − (c); (b) − (c′) (`graph - single4`) is secondary. Contradiction rate is the share of final answers with at least one
contradiction (lower is better).

In [10]:
METRICS = [("f1", "answer F1"), ("em", "exact match"), ("contra", "answers with a contradiction"),
           ("tokens", "tokens per question"), ("calls", "LLM calls per question"), ("searches", "searches per question")]
PAIRS = [("graph", "mp"), ("graph", "single"), ("mp", "single"), ("graph", "single4")]   # last one: secondary
SUMMARY = {}

def table(label, rows, metrics=METRICS):
    print(f"\n{label}  (n = {len(rows)})")
    print(f"{'metric':<30}" + "".join(f"{COLS[c]:>18}" for c in COLS) +
          "".join(f"{f'{a} - {b} [95% CI]':>30}" for a, b in PAIRS))
    for k, lab in metrics:
        means = {c: np.mean([r[c][k] for r in rows]) for c in COLS}
        diffs = {(a, b): paired_boot([r[b][k] for r in rows], [r[a][k] for r in rows]) for a, b in PAIRS}
        SUMMARY[(label, k)] = (len(rows), means, diffs)
        pct = k in ("f1", "em", "contra")
        fm = (lambda x: f"{x:.1%}") if pct else (lambda x: f"{x:,.1f}")
        fd = (lambda t: f"{t[0]:+.1%} [{t[1]:+.1%}, {t[2]:+.1%}]") if pct else \
             (lambda t: f"{t[0]:+,.0f} [{t[1]:+,.0f}, {t[2]:+,.0f}]")
        print(f"{lab:<30}" + "".join(f"{fm(means[c]):>18}" for c in COLS) + "".join(f"{fd(diffs[p]):>30}" for p in PAIRS))

for b, rows in ROWS.items():
    table(b, rows)
ALL = [r for rows in ROWS.values() for r in rows]
table("both benchmarks", ALL)


musique  (n = 1000)
metric                              (a) messages  (b) shared graph  (c) single agent   (c') 4 searches           graph - mp [95% CI]       graph - single [95% CI]          mp - single [95% CI]      graph - single4 [95% CI]


answer F1                                  47.3%             50.8%             38.8%             36.3%          +3.5% [+1.3%, +5.7%]        +12.0% [+9.6%, +14.4%]         +8.5% [+6.1%, +10.8%]       +14.5% [+12.0%, +17.1%]
exact match                                35.4%             38.9%             31.0%             28.5%          +3.5% [+1.2%, +5.9%]         +7.9% [+5.5%, +10.3%]          +4.4% [+2.0%, +6.7%]        +10.4% [+7.9%, +13.0%]
answers with a contradiction                9.8%              8.1%              4.1%              4.6%          -1.7% [-4.0%, +0.7%]          +4.0% [+2.0%, +6.1%]          +5.7% [+3.6%, +7.8%]          +3.5% [+1.4%, +5.6%]


tokens per question                     15,624.2          16,032.8           6,203.4           9,078.4             +409 [+330, +489]      +9,829 [+9,550, +10,109]       +9,421 [+9,150, +9,695]       +6,954 [+6,781, +7,123]


LLM calls per question                       8.0               8.0               3.3               4.5                   +0 [+0, +0]                   +5 [+5, +5]                   +5 [+5, +5]                   +4 [+3, +4]
searches per question                        4.0               4.0               3.0               4.0                   +0 [+0, +0]                   +1 [+1, +1]                   +1 [+1, +1]                   +0 [+0, +0]

2wikimultihopqa  (n = 1000)
metric                              (a) messages  (b) shared graph  (c) single agent   (c') 4 searches           graph - mp [95% CI]       graph - single [95% CI]          mp - single [95% CI]      graph - single4 [95% CI]
answer F1                                  74.6%             76.4%             72.6%             71.2%          +1.7% [+0.1%, +3.4%]          +3.7% [+1.9%, +5.6%]          +2.0% [+0.1%, +4.0%]          +5.2% [+3.3%, +7.1%]


exact match                                63.4%             65.9%             65.6%             64.1%          +2.5% [+0.5%, +4.6%]          +0.3% [-1.8%, +2.3%]          -2.2% [-4.4%, -0.1%]          +1.8% [-0.2%, +3.9%]


answers with a contradiction                7.2%              6.2%              9.0%              7.8%          -1.0% [-2.8%, +0.9%]          -2.8% [-4.9%, -0.8%]          -1.8% [-4.1%, +0.3%]          -1.6% [-3.7%, +0.5%]
tokens per question                     16,224.8          17,055.0           3,380.3           8,192.1             +830 [+716, +948]    +13,675 [+13,391, +13,962]    +12,844 [+12,568, +13,120]       +8,863 [+8,677, +9,053]
LLM calls per question                       8.0               8.0               2.1               4.1                   +0 [+0, +0]                   +6 [+6, +6]                   +6 [+6, +6]                   +4 [+4, +4]


... (3 outputs left out)

answers with a contradiction                8.5%              7.1%              6.6%              6.2%          -1.4% [-2.9%, +0.1%]          +0.6% [-0.8%, +2.1%]          +1.9% [+0.4%, +3.5%]          +0.9% [-0.5%, +2.4%]


tokens per question                     15,924.5          16,543.9           4,791.9           8,635.3             +619 [+549, +689]    +11,752 [+11,532, +11,969]    +11,133 [+10,925, +11,340]       +7,909 [+7,778, +8,043]


LLM calls per question                       8.0               8.0               2.7               4.3                   +0 [+0, +0]                   +5 [+5, +5]                   +5 [+5, +5]                   +4 [+4, +4]


searches per question                        4.0               4.0               2.5               4.0                   +0 [+0, +0]                   +1 [+1, +2]                   +1 [+1, +2]                   +0 [+0, +0]


### By hop count (MuSiQue) and question type (2Wiki)

In [11]:
for b, rows in ROWS.items():
    for g in sorted({r["group"] for r in rows}):
        table(f"{b} / {g}", [r for r in rows if r["group"] == g], metrics=[("f1", "answer F1"),
                                                                           ("contra", "answers with a contradiction")])


musique / 2-hop  (n = 518)
metric                              (a) messages  (b) shared graph  (c) single agent   (c') 4 searches           graph - mp [95% CI]       graph - single [95% CI]          mp - single [95% CI]      graph - single4 [95% CI]
answer F1                                  56.1%             59.8%             48.4%             45.0%          +3.7% [+0.8%, +6.5%]        +11.3% [+8.1%, +14.8%]         +7.7% [+4.6%, +10.9%]       +14.8% [+11.3%, +18.5%]
answers with a contradiction                7.5%              6.8%              1.5%              3.1%          -0.8% [-3.7%, +2.1%]          +5.2% [+2.9%, +7.5%]          +6.0% [+3.7%, +8.5%]          +3.7% [+1.2%, +6.2%]

musique / 3-hop  (n = 316)
metric                              (a) messages  (b) shared graph  (c) single agent   (c') 4 searches           graph - mp [95% CI]       graph - single [95% CI]          mp - single [95% CI]      graph - single4 [95% CI]


answer F1                                  44.7%             48.1%             34.7%             32.5%          +3.4% [-1.2%, +7.9%]        +13.5% [+9.1%, +18.0%]        +10.0% [+5.7%, +14.3%]       +15.6% [+10.9%, +20.4%]
answers with a contradiction                9.5%              9.5%              6.0%              6.3%          +0.0% [-4.4%, +4.4%]          +3.5% [-0.6%, +7.6%]          +3.5% [-0.3%, +7.3%]          +3.2% [-0.9%, +7.3%]

musique / 4-hop  (n = 166)
metric                              (a) messages  (b) shared graph  (c) single agent   (c') 4 searches           graph - mp [95% CI]       graph - single [95% CI]          mp - single [95% CI]      graph - single4 [95% CI]
answer F1                                  24.6%             27.6%             16.6%             16.2%          +3.0% [-1.6%, +7.6%]        +11.0% [+5.4%, +16.7%]         +8.0% [+2.9%, +13.2%]        +11.3% [+5.8%, +17.3%]
answers with a contradiction               17.5%              9.6%              

answer F1                                  56.8%             57.6%             48.6%             49.0%          +0.8% [-1.8%, +3.4%]         +9.0% [+6.2%, +12.0%]         +8.2% [+5.0%, +11.5%]         +8.6% [+5.7%, +11.7%]
answers with a contradiction                6.1%              5.3%              1.5%              3.6%          -0.7% [-3.6%, +2.2%]          +3.9% [+1.7%, +6.3%]          +4.6% [+2.2%, +7.0%]          +1.7% [-1.0%, +4.4%]

2wikimultihopqa / inference  (n = 108)
metric                              (a) messages  (b) shared graph  (c) single agent   (c') 4 searches           graph - mp [95% CI]       graph - single [95% CI]          mp - single [95% CI]      graph - single4 [95% CI]
answer F1                                  83.1%             79.8%             84.8%             81.0%          -3.3% [-9.5%, +2.7%]         -5.0% [-11.5%, +1.6%]          -1.7% [-6.9%, +3.5%]          -1.2% [-7.8%, +5.8%]
answers with a contradiction                5.6%              5.6%  

### Token budgets: was the single agent really given the same budget?

The budget per question is fixed by (a) and (b); the table shows how much of it the single agent used, and how the
two-agent tokens split between retriever, verifier and final answerer.

In [12]:
for cond, b in [(c, b) for c in ("single", "single4") for b in list(DATA) + ["both"]]:
    ks = [k for k in SINGLE if EP[k]["cond"] == cond and (b == "both" or EP[k]["bench"] == b)]
    bud = np.array([EP[k]["budget"] for k in ks]); used = np.array([EP[k]["tokens"] for k in ks])
    print(f"{COLS[cond]:<18}{b:<18}: budget {bud.mean():,.0f} tokens per question, used {used.mean():,.0f} "
          f"({used.sum() / bud.sum():.0%}); over budget on {np.mean(used > bud):.0%} of questions; "
          f"calls {np.mean([EP[k]['calls'] for k in ks]):.1f}, searches {np.mean([EP[k]['searches'] for k in ks]):.1f}")
for c in ("mp", "graph"):
    at = Counter()
    for k in MAS:
        if EP[k]["cond"] == c:
            at.update(EP[k]["agent_tokens"])
    n = sum(EP[k]["cond"] == c for k in MAS)
    print(f"{CONDS[c]:<18} tokens per question by agent: " + ", ".join(f"{a} {v / n:,.0f}" for a, v in at.items()))

(c) single agent  musique           : budget 15,829 tokens per question, used 6,203 (39%); over budget on 0% of questions; calls 3.3, searches 3.0
(c) single agent  2wikimultihopqa   : budget 16,640 tokens per question, used 3,380 (20%); over budget on 0% of questions; calls 2.1, searches 2.1
(c) single agent  both              : budget 16,234 tokens per question, used 4,792 (30%); over budget on 0% of questions; calls 2.7, searches 2.5
(c') 4 searches   musique           : budget 15,829 tokens per question, used 9,078 (57%); over budget on 0% of questions; calls 4.5, searches 4.0
(c') 4 searches   2wikimultihopqa   : budget 16,640 tokens per question, used 8,192 (49%); over budget on 0% of questions; calls 4.1, searches 4.0
(c') 4 searches   both              : budget 16,234 tokens per question, used 8,635 (53%); over budget on 0% of questions; calls 4.3, searches 4.0
(a) messages       tokens per question by agent: retriever 7,405, verifier 7,849, final 670
(b) shared graph   tokens 

### The shared graph: claims and their statuses

How many claims the retriever wrote, how the verifier marked them, and how many had a valid source passage. A refuted
claim is withheld from the final answerer in (b); in (a) the same kind of finding stays in the transcript.

In [13]:
for b in DATA:
    ks = [k for k in MAS if EP[k]["cond"] == "graph" and EP[k]["bench"] == b]
    st_counts = Counter(a["status"] for k in ks for _, a in EP[k]["G"].nodes(data=True) if a["kind"] == "claim")
    n_cl = sum(st_counts.values())
    sourced = sum(1 for k in ks for c in claims(EP[k]) if any(e["rel"] == "source" for *_, e in EP[k]["G"].out_edges(c, data=True)))
    deps = sum(1 for k in ks for *_, e in EP[k]["G"].edges(data=True) if e["rel"] == "builds_on")
    changed = sum(1 for k in ks for c in claims(EP[k]) if len({s for _, s in EP[k]["G"].nodes[c]["history"]}) > 1)
    no_sup = np.mean([not claims(EP[k], "supported") for k in ks])
    print(f"{b}: {n_cl / len(ks):.1f} claims per question ({sourced / max(n_cl, 1):.0%} with a valid source passage, "
          f"{deps / len(ks):.1f} builds-on edges); statuses " +
          ", ".join(f"{s} {v / max(n_cl, 1):.0%}" for s, v in st_counts.most_common()) +
          f"; status changed by a later round: {changed}; questions with no supported claim: {no_sup:.0%}")

musique: 3.7 claims per question (98% with a valid source passage, 2.2 builds-on edges); statuses supported 90%, open 7%, refuted 3%; status changed by a later round: 133; questions with no supported claim: 2%
2wikimultihopqa: 3.6 claims per question (100% with a valid source passage, 2.7 builds-on edges); statuses supported 96%, refuted 2%, open 2%; status changed by a later round: 60; questions with no supported claim: 0%


### One question, three episodes

The first MuSiQue question on which the three conditions do not all get the same F1 (an illustration, not a typical case).

In [14]:
b = "musique"
qi = next((i for i, r in enumerate(ROWS[b]) if len({r[c]["f1"] for c in CONDS}) > 1), 0)
q = DATA[b]["questions"][qi]
print(f"Q: {q['question']}\nGold: {q['golds'][0]}\n")
st = EP[f"mp|{b}|{qi}"]
print("(a) MESSAGES\n" + transcript(st)[:3000])
print(f"-> {st['final']}  F1 {ROWS[b][qi]['mp']['f1']:.2f}\n")
st = EP[f"graph|{b}|{qi}"]
print("(b) SHARED GRAPH\n" + graph_view(st)[:3000])
print(f"-> {st['final']}  F1 {ROWS[b][qi]['graph']['f1']:.2f}\n")
st = EP[f"single|{b}|{qi}"]
print("(c) SINGLE AGENT notes\n" + "\n".join(f"[step {s}] {n}" for s, n in st["notes"])[:3000])
print(f"-> {st['final']}  F1 {ROWS[b][qi]['single']['f1']:.2f}; budget {st['budget']:,.0f}, used {st['tokens']:,}\n")
st = EP[f"single4|{b}|{qi}"]
print(f"(c') SINGLE AGENT, 4 searches required -> {st['final']}  F1 {ROWS[b][qi]['single4']['f1']:.2f}; "
      f"searches {st['searches']}, used {st['tokens']:,} tokens")
for c in COLS:
    v = VERDICT[f"{c}|{b}|{qi}"]
    print(f"judge {COLS[c]}: {v}")

Q: When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona?
Gold: June 1982

(a) MESSAGES
Messages so far:
[Retriever, round 1] The passages do not identify the person whose Copa del Rey goals were compared with Messi’s, or state when that person was signed by Barcelona. [P1–P4]

[Retriever, round 2] The passages still do not identify the person whose Copa del Rey goals were compared with Messi’s, nor give the date that person was signed by Barcelona. The available Barcelona signing dates concern Diego Maradona and László Kubala, but nothing here connects either to the comparison [P3, P7].

[Verifier, round 1] Round 1 finding: supported — none of the passages identifies the person whose Copa del Rey goals were compared with Messi’s or gives that person’s Barcelona signing date. Round 2 finding: supported — P3 and P7 give signing information for Maradona and Kubala, but do not connect either to the stated comparison. Missing fact: the identity of the pe

## What this run cost

Only this notebook's stages; the passage and question embeddings are Notebook 1.4a's (cost there). The forecast scales the
test run's calls and tokens per question to 2,000 questions at the Batch price (half the standard price used in test
mode) and compares it with `paper/v2/cost_forecast.py` (5.5: $6.88).

In [15]:
LEDGER = llm.ledger(STAGES)
print(f"{'stage':<30}{'calls':>8}{'input tok':>13}{'output tok':>12}{'USD':>8}{'via Batch':>11}")
for s in STAGES:
    a = LEDGER[s]
    print(f"{s:<30}{a['calls']:>8,}{a['in']:>13,}{a['out']:>12,}{a['usd']:>8.3f}{a['batch_share']:>11.0%}")
emb_tok = sum(json.load(open(f"{llm.cache}/{e}.embed.json"))["tokens"] for e in set(EMBEDS))
llm_usd = sum(LEDGER[s]["usd"] for s in STAGES)
print(f"{'query embeddings':<30}{'':>8}{emb_tok:>13,}{'':>12}{emb_tok * EMBED_PRICE / 1e6:>8.3f}")
print(f"{'TOTAL (this notebook)':<30}{sum(LEDGER[s]['calls'] for s in STAGES):>8,}"
      f"{sum(LEDGER[s]['in'] for s in STAGES):>13,}{sum(LEDGER[s]['out'] for s in STAGES):>12,}"
      f"{llm_usd + emb_tok * EMBED_PRICE / 1e6:>8.3f}")
n_q = sum(len(d["questions"]) for d in DATA.values())
if TEST:
    scale = 2000 / n_q
    full = llm_usd * scale / 2                       # test = standard tier; full run = Batch (half price)
    print(f"\nForecast of the full run (2,000 questions, Batch): {sum(LEDGER[s]['calls'] for s in STAGES) * scale:,.0f} "
          f"calls, ${full:.2f} (+ ${emb_tok * scale * EMBED_PRICE / 1e6:.2f} query embeddings); "
          f"cost_forecast.py: 50,000 calls, $6.88")

stage                            calls    input tok  output tok     USD  via Batch
nb29_mas_r1                      4,000    3,473,524     306,954   0.250       100%
nb29_mas_r2                      4,000    5,960,571     245,671   0.359       100%
nb29_mas_v1                      4,000    8,362,747     452,629   0.531       100%
nb29_mas_r3                      4,000    8,770,171     203,463   0.489       100%
nb29_mas_v2                      4,000   11,066,742     435,819   0.662       100%
nb29_mas_r4                      4,000   11,317,351     188,311   0.613       100%
nb29_mas_v3                      4,000   11,405,852     443,847   0.681       100%
nb29_mas_final                   4,000    2,105,443     197,687   0.155       100%
nb29_single_s1                   2,000    1,802,762     198,974   0.140       100%
nb29_single_s2                   1,465    2,243,150     158,751   0.152       100%
nb29_single_s3                     842    1,830,748      82,788   0.112       100%
nb29

## Limitations

- **Equal budget is an upper bound for the single agent, not a forced spend.** The single agent may answer before its
  budget is used; its tokens are reported next to its budget. Tran & Kiela (2026) control *thinking* tokens of
  reasoning models; here the model runs without hidden reasoning and the budget counts all input and output tokens,
  which is what the API bills.
- **Same searches, not the same passages.** Every condition gets at most 4 searches of 4 passages; the queries differ, so
  the evidence each condition saw differs. Passages retrieved and searches used are reported.
- **The final answerer of (a) and (b) does not see the passages.** This isolates the channel (transcript vs. graph);
  a design in which the final answerer also reads the passages would test something else.
- **Contradiction judge.** One LLM judge (the same model as the agents), checked against retrieved passages only; the
  second-judge check (`second_judge.py`, gpt-4.1, 500 items) should be run on `nb29_contradiction_judge`.
- **One configuration.** One model, one prompt per role, one schedule (4 retriever rounds, 3 verifier rounds), run once.
- **The graph is small.** A claim graph per question has tens of nodes; the result says nothing about large shared
  graphs that many agents write over many tasks.

**References.** Tran & Kiela, *Single-Agent LLMs Outperform Multi-Agent Systems on Multi-Hop Reasoning Under Equal
Thinking Token Budgets*, arXiv 2604.02460 (2026) · Salemi et al., *LLM-based Multi-Agent Blackboard System for
Information Discovery in Data Science*, arXiv 2510.01285 · Han & Zhang, *Exploring Advanced LLM Multi-Agent Systems
Based on Blackboard Architecture*, arXiv 2507.01701 · Trivedi et al., *MuSiQue*, TACL 2022 · Ho et al.,
*2WikiMultiHopQA*, COLING 2020.

In [16]:
print("=" * 120)
print(f"RESULTS SUMMARY — Notebook 4.5: shared evidence graph vs. messages vs. single agent, {MODE} run, gpt-6-luna "
      f"(no reasoning), MuSiQue + 2Wiki")
print("=" * 120)
print(f"{'benchmark':<20}{'metric':<30}" + "".join(f"{COLS[c]:>18}" for c in COLS) +
      "".join(f"{f'{a} - {b} [95% CI]':>30}" for a, b in PAIRS))
for label in list(DATA) + ["both benchmarks"]:
    for k, lab in METRICS:
        if k not in ("f1", "contra", "tokens"):
            continue
        n, means, diffs = SUMMARY[(label, k)]
        pct = k != "tokens"
        fm = (lambda x: f"{x:.1%}") if pct else (lambda x: f"{x:,.0f}")
        fd = (lambda t: f"{t[0]:+.1%} [{t[1]:+.1%}, {t[2]:+.1%}]") if pct else \
             (lambda t: f"{t[0]:+,.0f} [{t[1]:+,.0f}, {t[2]:+,.0f}]")
        print(f"{label:<20}{lab:<30}" + "".join(f"{fm(means[c]):>18}" for c in COLS) +
              "".join(f"{fd(diffs[p]):>30}" for p in PAIRS))
mb = [r for r in ROWS["musique"]]
print("\nMuSiQue F1 by hop, (b) - (a) and (b) - (c): " + "; ".join(
    f"{g} " + ", ".join(f"{t[0]:+.1%} [{t[1]:+.1%}, {t[2]:+.1%}]" for t in
                        (SUMMARY[(f'musique / {g}', 'f1')][2][("graph", "mp")], SUMMARY[(f'musique / {g}', 'f1')][2][("graph", "single")]))
    for g in sorted({r["group"] for r in mb})))
def verdict(t, higher_better=True):
    lo, hi = (t[1], t[2]) if higher_better else (-t[2], -t[1])
    return "better" if lo > 0 else "worse" if hi < 0 else "no resolvable difference"
n, means, diffs = SUMMARY[("both benchmarks", "f1")]
_, _, cd = SUMMARY[("both benchmarks", "contra")]
half = (diffs[("graph", "mp")][2] - diffs[("graph", "mp")][1]) / 2
print(f"\nVerdict on F1 (primary), both benchmarks, n = {n}: graph vs messages {verdict(diffs[('graph', 'mp')])}; "
      f"graph vs single agent {verdict(diffs[('graph', 'single')])}; messages vs single agent {verdict(diffs[('mp', 'single')])}")
print(f"Verdict on contradictions (fewer is better): graph vs messages {verdict(cd[('graph', 'mp')], False)}; "
      f"graph vs single agent {verdict(cd[('graph', 'single')], False)}")
print(f"Secondary (added after the test run): graph vs single agent with 4 searches required - F1 "
      f"{verdict(diffs[('graph', 'single4')])}, contradictions {verdict(cd[('graph', 'single4')], False)}")
print(f"Half-width of the (b) - (a) F1 interval: {half:.1%} (protocol target: below 5 points at the full n)")
print("\nLecture message (Task 4.5): a shared graph of claims lets a verifier's verdicts persist and filters refuted claims"
      "\nout of the final answer. Whether that beats message passing, and a single agent with the same token budget, is"
      "\nwhat the verdict lines say.")

RESULTS SUMMARY — Notebook 4.5: shared evidence graph vs. messages vs. single agent, batch run, gpt-6-luna (no reasoning), MuSiQue + 2Wiki
benchmark           metric                              (a) messages  (b) shared graph  (c) single agent   (c') 4 searches           graph - mp [95% CI]       graph - single [95% CI]          mp - single [95% CI]      graph - single4 [95% CI]
musique             answer F1                                  47.3%             50.8%             38.8%             36.3%          +3.5% [+1.3%, +5.7%]        +12.0% [+9.6%, +14.4%]         +8.5% [+6.1%, +10.8%]       +14.5% [+12.0%, +17.1%]
musique             answers with a contradiction                9.8%              8.1%              4.1%              4.6%          -1.7% [-4.0%, +0.7%]          +4.0% [+2.0%, +6.1%]          +5.7% [+3.6%, +7.8%]          +3.5% [+1.4%, +5.6%]
musique             tokens per question                       15,624            16,033             6,203             9,078          